# Experiment 2 — Articulatory PWCCA Probing

*Paper Sec 2.4. Reproduces Fig 3 (per-codec PWCCA across layers) and Fig 4 (MIMI's WavLM-distilled first layer vs accumulated acoustic).*

For every track in the 75-Speaker rtMRI corpus, we compare the codec's per-layer hidden states with the 120-gridline Vocal Tract Distance (VTD) extracted from the corresponding rtMRI sequence, using **Projection-Weighted CCA**.

Pre-computed JSON caches under `data/output_stats_cache/` let you skip the GPU loop entirely. Pre-computed VTD `.npy` files under `data/output_vtd/` come from `scripts/compute_vtd_batch.py`.

In [ ]:
import os, sys, pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Paths

In [ ]:
VTD_DIR    = os.path.join(REPO_ROOT, 'data', 'output_vtd')
SPEECH_DIR = '/data/xuanshi/DATA/SPAN/SPAN'  # parent of <sub_id>/2drt/audio/<trackname>_audio.wav
CACHE_DIR  = os.path.join(REPO_ROOT, 'data', 'output_stats_cache')
os.makedirs(CACHE_DIR, exist_ok=True)
print('VTD files:', len([p for p in os.listdir(VTD_DIR) if p.endswith('_vtd.npy')]) if os.path.isdir(VTD_DIR) else 'directory missing')

## 2. (Optional) recompute — heavy GPU loop

In [ ]:
from src.codecs import load_encodec, load_dac, load_mimi, load_mimo
from src.analysis import compute_codec_vs_vtd_similarity

RECOMPUTE = []  # e.g. ['mimi']; empty means just load caches
for name in RECOMPUTE:
    factory = {
        'encodec': lambda: load_encodec(bandwidth=12.0, device=DEVICE),
        'dac':     lambda: load_dac(device=DEVICE),
        'mimi':    lambda: load_mimi(device=DEVICE),
        'mimo':    lambda: load_mimo(checkpoint='/data/xuanshi/REPO/vocal_tract_distance/lib/MiMo_Audio_Tokenizer/MiMo-Audio-Tokenizer', device=DEVICE),
    }[name]
    codec = factory()
    cache_path = os.path.join(CACHE_DIR, f'cca_similarity_across_layers_{codec.name}.json')
    compute_codec_vs_vtd_similarity(codec, VTD_DIR, SPEECH_DIR, cache_path=cache_path, use_cca=True, use_rsa=False)
    del codec; torch.cuda.empty_cache()

## 3. Plot Fig 3 — per-codec PWCCA curves

In [ ]:
import json, glob
from src.analysis import plot_similarity
caches = sorted(glob.glob(os.path.join(CACHE_DIR, 'cca_*.json')))
if not caches:
    print('no caches; set RECOMPUTE or place pre-computed JSONs in', CACHE_DIR)
else:
    fig, axes = plt.subplots(1, len(caches), figsize=(4*len(caches), 3.5), squeeze=False)
    for ax, path in zip(axes.flat, caches):
        d = json.load(open(path))
        plot_similarity(d.get('cca_means'), d.get('rsa_means'), d['num_layers'], d['codec_model'], ax=ax)
    plt.tight_layout(); plt.show()

## 4. Fig 4 — MIMI semantic vs accumulated acoustic

With `add_semantic_to_acoustic=False`, MIMI returns the WavLM-distilled first layer separately from the cumulative-sum acoustic layers.

In [ ]:
RUN_FIG4 = False
if RUN_FIG4:
    codec = load_mimi(device=DEVICE, add_semantic_to_acoustic=False)
    cache_path = os.path.join(CACHE_DIR, 'cca_similarity_across_layers_mimi_separate.json')
    compute_codec_vs_vtd_similarity(codec, VTD_DIR, SPEECH_DIR, cache_path=cache_path, use_cca=True, use_rsa=False)
    del codec; torch.cuda.empty_cache()
fig_path = os.path.join(CACHE_DIR, 'cca_similarity_across_layers_mimi_separate.json')
if os.path.exists(fig_path):
    d = json.load(open(fig_path))
    fig, ax = plt.subplots(figsize=(5, 3))
    plot_similarity(d.get('cca_means'), d.get('rsa_means'), d['num_layers'], 'mimi (semantic vs acoustic)', ax=ax)
    plt.show()
else:
    print('Set RUN_FIG4=True to populate this cache.')